In [2]:
import json
import pandas as pd
from pathlib import Path
import numpy as np

from src.data_export import export_points, export_bmarkers

In [ ]:
# load all the excel files from the directory and combine
root = Path("./raw_data")
dfs = []
for file in root.rglob("*.xlsx"):
    df = pd.read_excel(file, engine="openpyxl")
    df["source_file"] = str(file.relative_to(root))
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)
combined_df
# expected column layout -> Name |	ID |	Position X |	Position Y |	ROW |


In [ ]:
# identify all areas in the WH -> waypoints has area as a ID
areas = combined_df.query("Name == 'WAYPOINT'")["ID"].to_list()
areas

In [ ]:
# calculate origin and max for a plot
corners_df = (combined_df.groupby('source_file', as_index=False)[['Position X', 'Position Y']].agg(['min', 'max']))
corners_df

In [ ]:
# transfer origin and max to a dict (json)
corners_dict = {}

for  inx, row in corners_df.iterrows():
	corners_dict[areas[inx]] = {
        "corners": [
            {"x": int(row[('Position X', 'min')]), "y": int(row[('Position Y', 'min')]), "label": "Origin"},
            {"x": int(row[('Position X', 'max')])+1, "y": int(row[('Position Y', 'max')])+1, "label": "Max"}
            ]}

corners_dict

In [ ]:
# prepare bmarkers in a dict
bmarkers_df = combined_df[combined_df["Name"] == 'BMARKER']
bmarkers_df['unique_rack'] = bmarkers_df["ID"].apply(lambda x: x.split(".")[0])

In [ ]:
# export points, combine with other dits and dump to a json file
for area in areas:
    area_bms = bmarkers_df[bmarkers_df["source_file"].str.endswith(f"{area}.xlsx")]
    bmarkers_dict = export_bmarkers(area_bms,area)

    area_points = export_points(combined_df,area)

    combined = {**corners_dict[area], **bmarkers_dict, **area_points}
    with open(f"json_files/mapping_{area}.json", "w", encoding="utf-8") as f:
        json.dump(combined, f, ensure_ascii=False, indent=2)


In [50]:
# export all point from reports i distances
res_ok = pd.DataFrame(columns={'x':float, 'y':float, 'description':str, 'is_key':bool, 'area_id':str})
dist_st = set()
reports = Path("./reports")
for file in reports.rglob("*.pkl"):
	area_id = str(file.relative_to(reports)).split('.')[0].replace('mapping', '').replace('_','')
	point_dict = pd.read_pickle(file)
	st = set()
	for j,i in enumerate(point_dict.keys()):
		# x,y, description, is_key, area
		st.add((point_dict[i]['waypoints'][0][0], point_dict[i]['waypoints'][0][1], i[0], 'WP' in str(i[0]), area_id.upper()))
		st.add((point_dict[i]['waypoints'][-1][0], point_dict[i]['waypoints'][-1][1], i[1], 'WP' in str(i[1]), area_id.upper()))
		dist_st.add((f"{area_id.upper()}_{i[0]}", f"{area_id.upper()}_{i[1]}", round(point_dict[i]['distance'],2)))
	if len(res_ok) == 0:
		res_ok = pd.DataFrame(data=st, columns=res_ok.columns)
	else:
		res_ok= pd.concat([res_ok, pd.DataFrame(data=st,columns=res_ok.columns)])

In [53]:
distances = pd.DataFrame(data = dist_st, columns={'from':str, 'to':str, 'measure':float})
distances.loc[:, 'from'] = distances['from'].str.replace(r'^PP_', '', regex=True)
distances.loc[:, 'to']   = distances['to'].str.replace(r'^PP_', '', regex=True)
distances.to_pickle('reports/distances_db.pkl')

In [34]:
res_ok.loc[:, 'description'] = np.where(
      ~res_ok['is_key'],
      res_ok['area_id'] + '_' + res_ok['description'],
      res_ok['description']
 )

In [39]:
res_ok = res_ok.query("not (area_id == 'PP' and description != 'WP_PP')")

,x,y,description,is_key,area_id
4,17.68,21.78,WP_PP,True,PP
0,8.01,31.93,CD1_F13,False,CD1
1,1.24,22.02,CD1_A7,False,CD1
2,4.32,22.48,CD1_C5,False,CD1
3,1.24,32.02,CD1_A17,False,CD1
...,...,...,...,...,...
75,15.06,9.43,CD2_J9,False,CD2
76,23.45,10.11,CD2_N8,False,CD2
77,19.26,5.75,CD2_L5,False,CD2
78,19.26,7.63,CD2_L7,False,CD2


In [40]:
res_ok.to_pickle("reports/all_points_db.pkl")

In [45]:
# export distances
reports = Path("./reports")
g = pd.read_pickle(reports / 'm_mapping.pkl')
for i in g.keys():
	print(i, round(g[i]['distance'],3))

('WP_M', 'A10') 10.398
('WP_M', 'D8') 11.373
('WP_M', 'E8') 14.169
('WP_M', 'F10') 15.681
('WP_M', 'G10') 18.692
('WP_M', 'A3') 2.72
('WP_M', 'A1') 0.798
('WP_M', 'A2') 1.737
('WP_M', 'A4') 3.712
('WP_M', 'A7') 7.151
('WP_M', 'A6') 5.704
('WP_M', 'A8') 8.15
('WP_M', 'A9') 9.149
('WP_M', 'A5') 4.707
('WP_M', 'B1') 1.785
('WP_M', 'B2') 2.765
('WP_M', 'B3') 3.756
('WP_M', 'B4') 4.75
('WP_M', 'B5') 5.747
('WP_M', 'B7') 7.743
('WP_M', 'B8') 8.741
('WP_M', 'B9') 9.74
('WP_M', 'B6') 6.744
('WP_M', 'C4') 6.683
('WP_M', 'C2') 4.684
('WP_M', 'C3') 5.683
('WP_M', 'C5') 7.682
('WP_M', 'C1') 3.686
('WP_M', 'C7') 9.682
('WP_M', 'C8') 10.682
('WP_M', 'C9') 11.682
('WP_M', 'C6') 8.682
('WP_M', 'D2') 4.814
('WP_M', 'D3') 5.771
('WP_M', 'D4') 6.749
('WP_M', 'D1') 3.744
('WP_M', 'D6') 9.134
('WP_M', 'D7') 10.128
('WP_M', 'D5') 7.736
('WP_M', 'E7') 12.919
('WP_M', 'E6') 11.919
('WP_M', 'E5') 10.51
('WP_M', 'E4') 9.51
('WP_M', 'E3') 8.51
('WP_M', 'E2') 7.511
('WP_M', 'E1') 6.514
('WP_M', 'F1') 6.795
('WP_M